In [1]:
import pandas as pd
from pathlib import Path
import os

print("CWD:", os.getcwd())
print("data/raw contents:", os.listdir("data/raw"))
data_dir = Path("data/raw")
summary = []

for file in data_dir.glob("*_price.csv"):
    ticker = file.stem.split("_")[0]
    df = pd.read_csv(file, parse_dates=["Date"])
    miss_pct = df.isna().mean() * 100
    for col, pct in miss_pct.items():
        summary.append(["Price", ticker, col, round(pct,2)])

fund_file = data_dir / "tech_fundamentals.csv"
df_fund = pd.read_csv(fund_file)
miss_pct = df_fund.isna().mean() * 100
for col, pct in miss_pct.items():
    summary.append(["Fundamentals", "", col, round(pct,2)])

missing_df = pd.DataFrame(summary, columns=["Dataset","Ticker","Column","Missing_%"])
print(missing_df)


CWD: /Users/liwenchen/Desktop/Innovation AI
data/raw contents: ['AMZN_price.csv', 'GOOGL_price.parquet', 'AAPL_price.parquet', 'MSFT_price.csv', 'AMZN_price.parquet', 'MSFT_price.parquet', 'ORCL_price.csv', 'META_price.parquet', 'tech_fundamentals.csv', 'IBM_price.csv', 'TSLA_price.csv', 'NVDA_price.parquet', 'NVDA_price.csv', 'META_price.csv', 'IBM_price.parquet', 'GOOGL_price.csv', 'AAPL_price.csv', 'tech_fundamentals.parquet', 'CRM_price.parquet', 'TSLA_price.parquet', 'CRM_price.csv', 'ORCL_price.parquet']
         Dataset Ticker                     Column  Missing_%
0          Price   AMZN                       Date       0.00
1          Price   AMZN                       Open       0.06
2          Price   AMZN                       High       0.06
3          Price   AMZN                        Low       0.06
4          Price   AMZN                      Close       0.06
..           ...    ...                        ...        ...
60  Fundamentals                            Ticker

In [2]:
import pandas as pd
from pathlib import Path

data_dir = Path("data/raw")
price_files = list(data_dir.glob("*_price.csv"))
tickers = [f.stem.split("_")[0] for f in price_files]

price_dfs = {
    t: pd.read_csv(data_dir / f"{t}_price.csv", parse_dates=["Date"])
    for t in tickers
}

for t, df in price_dfs.items():
    # Modern forward and backward fill for missing values
    if df.isna().sum().sum() > 0:
        df.ffill(inplace=True)
        df.bfill(inplace=True)

outlier_summary = []
for t, df in price_dfs.items():
    num_cols = df.select_dtypes(include="number").columns
    for col in num_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        mask = (df[col] < lower) | (df[col] > upper)
        n_out = mask.sum()
        pct_out = round(n_out / len(df) * 100, 2)
        outlier_summary.append({"Ticker": t, "Column": col, "Outliers": n_out, "Pct_Outliers": pct_out})
        df[col] = df[col].clip(lower, upper)

outlier_df = pd.DataFrame(outlier_summary)
print("Outlier Summary (after winsorization):")
print(outlier_df.to_string(index=False))

Outlier Summary (after winsorization):
Ticker Column  Outliers  Pct_Outliers
  AMZN   Open         0          0.00
  AMZN   High         0          0.00
  AMZN    Low         0          0.00
  AMZN  Close         0          0.00
  AMZN Volume        72          4.39
  MSFT   Open         0          0.00
  MSFT   High         0          0.00
  MSFT    Low         0          0.00
  MSFT  Close         0          0.00
  MSFT Volume        95          5.80
  ORCL   Open        17          1.04
  ORCL   High        18          1.10
  ORCL    Low        18          1.10
  ORCL  Close        18          1.10
  ORCL Volume        98          5.98
   IBM   Open       183         11.17
   IBM   High       185         11.29
   IBM    Low       176         10.74
   IBM  Close       181         11.04
   IBM Volume       103          6.28
  TSLA   Open         0          0.00
  TSLA   High         0          0.00
  TSLA    Low         0          0.00
  TSLA  Close         0          0.00
  TSLA Volu

In [3]:
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler

fundamentals_path = Path("data/raw/tech_fundamentals.csv")
fund_df = pd.read_csv(fundamentals_path)
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

price_scaled = {}
for ticker, df in price_dfs.items():
    # Only select numeric columns (exclude 'Date' and any non-numeric columns)
    numeric_cols = df.select_dtypes(include="number").columns
    scaler = StandardScaler()
    arr = scaler.fit_transform(df[numeric_cols])
    df_std = pd.DataFrame(arr, index=df.index, columns=numeric_cols)
    
    # Optionally, add back 'Date' and other non-numeric columns
    if 'Date' in df.columns:
        df_std['Date'] = df['Date'].values
        cols = ['Date'] + list(numeric_cols)
        df_std = df_std[cols]
    
    price_scaled[ticker] = df_std
    df_std.to_parquet(output_dir / f"{ticker}_price_std.parquet", index=False)

# Fundamentals block (your code is correct, since only numeric cols are passed)
fund_cols = ["EPS (ttm)", "P/E (ttm)", "Revenue (ttm)", "Operating Cash Flow (ttm)"]
scaler_fund = StandardScaler()
fund_arr = scaler_fund.fit_transform(fund_df[fund_cols])
fund_std = fund_df.copy()
fund_std[fund_cols] = fund_arr
fund_std.to_parquet(output_dir / "tech_fundamentals_std.parquet", index=False)

print("AAPL price (standardized) head:")
print(price_scaled["AAPL"].head())
print("\nFundamentals (standardized) head:")
print(fund_std.head())


AAPL price (standardized) head:
        Date      Open      High       Low     Close    Volume
0 2019-01-02 -1.847015 -1.841140 -1.840344 -1.835212  1.460747
1 2019-01-03 -1.893037 -1.896007 -1.892398 -1.901505  2.467339
2 2019-01-04 -1.890717 -1.884181 -1.884737 -1.875924  2.467339
3 2019-01-07 -1.873127 -1.883011 -1.875799 -1.877314  2.467339
4 2019-01-08 -1.869499 -1.870517 -1.864647 -1.865430  1.849343

Fundamentals (standardized) head:
  Ticker  EPS (ttm)  P/E (ttm)  Revenue (ttm)  Operating Cash Flow (ttm)
0   AAPL  -0.265073  -0.474758       0.940038                   0.768914
1   MSFT   0.736825  -0.339357       0.239746                   1.206281
2  GOOGL   0.123701  -0.793944       0.721644                   1.245378
3   AMZN  -0.308099  -0.399240       2.282789                   0.858790
4   META   2.677619  -0.586818      -0.295588                   0.490871


In [4]:
import pandas as pd
from pathlib import Path

processed_dir = Path("data/processed")
price_scaled = {
    p.stem.split("_")[0]: pd.read_parquet(p)
    for p in processed_dir.glob("*_price_std.parquet")
}
fund_std = pd.read_parquet(processed_dir / "tech_fundamentals_std.parquet")

panel_list = []
for ticker, df in price_scaled.items():
    df_copy = df.copy()
    df_copy['Ticker'] = ticker
    fund_row = fund_std.loc[fund_std['Ticker'] == ticker].iloc[0]
    for col in ['EPS (ttm)', 'P/E (ttm)', 'Revenue (ttm)', 'Operating Cash Flow (ttm)']:
        df_copy[col] = fund_row[col]
    panel_list.append(df_copy)

panel_df = pd.concat(panel_list)

# Remove duplicate 'Date' column if exists
if 'Date' in panel_df.columns and panel_df.index.name == 'Date':
    panel_df = panel_df.reset_index().drop(columns=['Date'])
elif panel_df.index.name == 'Date':
    panel_df = panel_df.reset_index()

output_file_parquet = processed_dir / "tech_panel.parquet"
output_file_csv = processed_dir / "tech_panel.csv"
panel_df.to_parquet(output_file_parquet, index=False)
panel_df.to_csv(output_file_csv, index=False)

print(panel_df.head())
print(panel_df.dtypes)


        Date      Open      High       Low     Close    Volume Ticker  \
0 2019-01-02 -1.847015 -1.841140 -1.840344 -1.835212  1.460747   AAPL   
1 2019-01-03 -1.893037 -1.896007 -1.892398 -1.901505  2.467339   AAPL   
2 2019-01-04 -1.890717 -1.884181 -1.884737 -1.875924  2.467339   AAPL   
3 2019-01-07 -1.873127 -1.883011 -1.875799 -1.877314  2.467339   AAPL   
4 2019-01-08 -1.869499 -1.870517 -1.864647 -1.865430  1.849343   AAPL   

   EPS (ttm)  P/E (ttm)  Revenue (ttm)  Operating Cash Flow (ttm)  
0  -0.265073  -0.474758       0.940038                   0.768914  
1  -0.265073  -0.474758       0.940038                   0.768914  
2  -0.265073  -0.474758       0.940038                   0.768914  
3  -0.265073  -0.474758       0.940038                   0.768914  
4  -0.265073  -0.474758       0.940038                   0.768914  
Date                         datetime64[ns]
Open                                float64
High                                float64
Low                  

In [5]:
import pandas as pd
from pathlib import Path

raw_dir = Path("data/raw")
processed_dir = Path("data/processed")
report = []

for file in raw_dir.glob("*_price.csv"):
    ticker = file.stem.split("_")[0]
    df_raw = pd.read_csv(file, parse_dates=["Date"])
    df_clean = pd.read_parquet(processed_dir / f"{ticker}_price_std.parquet")
    n = len(df_raw)

    for col in ["Open", "High", "Low", "Close", "Volume"]:
        miss_before = df_raw[col].isna().sum()
        miss_after = df_clean[col].isna().sum()
        miss_before_pct = miss_before / n * 100
        miss_after_pct = miss_after / n * 100
        Q1 = df_raw[col].quantile(0.25)
        Q3 = df_raw[col].quantile(0.75)
        IQR = Q3 - Q1
        lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
        out_n = ((df_raw[col] < lower) | (df_raw[col] > upper)).sum()
        out_pct = out_n / n * 100

        report.append({
            "Dataset": "Price",
            "Ticker": ticker,
            "Column": col,
            "Miss_before_%": round(miss_before_pct, 2),
            "Miss_after_%": round(miss_after_pct, 2),
            "Outliers_%": round(out_pct, 2)
        })

df_raw_f = pd.read_csv(raw_dir / "tech_fundamentals.csv")
df_clean_f = pd.read_parquet(processed_dir / "tech_fundamentals_std.parquet")
m = len(df_raw_f)

for col in ["EPS (ttm)", "P/E (ttm)", "Revenue (ttm)", "Operating Cash Flow (ttm)"]:
    miss_before_pct = df_raw_f[col].isna().mean() * 100
    miss_after_pct = df_clean_f[col].isna().mean() * 100
    report.append({
        "Dataset": "Fundamentals",
        "Ticker": "",
        "Column": col,
        "Miss_before_%": round(miss_before_pct, 2),
        "Miss_after_%": round(miss_after_pct, 2),
        "Outliers_%": None
    })

report_df = pd.DataFrame(report)
print(report_df)


         Dataset Ticker                     Column  Miss_before_%  \
0          Price   AMZN                       Open           0.06   
1          Price   AMZN                       High           0.06   
2          Price   AMZN                        Low           0.06   
3          Price   AMZN                      Close           0.06   
4          Price   AMZN                     Volume           0.06   
5          Price   MSFT                       Open           0.00   
6          Price   MSFT                       High           0.00   
7          Price   MSFT                        Low           0.00   
8          Price   MSFT                      Close           0.00   
9          Price   MSFT                     Volume           0.00   
10         Price   ORCL                       Open           0.00   
11         Price   ORCL                       High           0.00   
12         Price   ORCL                        Low           0.00   
13         Price   ORCL           